# Lesson 2.2 — 数字 PDF 文本提取（含多栏处理）


**前提**：PDF 已确认为数字版（见 [01_detect_pdf_type.ipynb](01_detect_pdf_type.ipynb)）。

**目标**：从数字 PDF 中可靠提取文本，并处理多栏布局、页眉页脚等常见坑点。

---

本课对比三种方法：

| 方法 | 工具 | 特点 | 适用场景 |
|------|------|------|----------|
| **A** | PyPDF2 | 最简单，质量一般 | 快速原型、零依赖 |
| **B** | PyMuPDF | 速度快，质量好 | **默认首选** |
| **C** | pdfplumber + 多栏检测 | 版面感知 | 两栏/多栏论文 |

## 数字 PDF 抽文本：三种姿势

### Level 1: 暴力提取（PyPDF2）
```python
from PyPDF2 import PdfReader
reader = PdfReader("file.pdf")
text = "\n".join(p.extract_text() for p in reader.pages)
```
✅ 最快上手  ❌ 多栏交错、表格乱、中文偶发乱码

### Level 2: 版面感知（PyMuPDF）
```python
import fitz
doc = fitz.open("file.pdf")
for page in doc:
    text = page.get_text(sort=True)  # 按视觉顺序排序
```
✅ 大多数情况够用  ⚠️ 复杂多栏仍可能出错

### Level 3: 自己控制 bbox（pdfplumber）
```python
with pdfplumber.open("file.pdf") as pdf:
    words = pdf.pages[0].extract_words()  # 每个词带 (x0,y0,x1,y1)
    # 自己按 x0 分栏，按 y0 排序
```
✅ 完全可控  ❌ 代码量大

## 工具对比实验：同一份论文，三种工具

| 工具 | 字符数 | 多栏正确 | 表格识别 | 公式 | 耗时 |
|------|--------|---------|----------|------|------|
| PyPDF2 | 12,400 | ❌（栏交错） | ❌ | ❌（变乱码） | 0.3s |
| PyMuPDF | 13,200 | ⚠️（需 sort） | ⚠️（合并成文本） | ⚠️ | 0.1s |
| pdfplumber | 13,800 | ✅（手写） | ✅ | ⚠️ | 2.1s |
| LlamaParse | 14,200 | ✅ | ✅（→Markdown） | ✅ | 6s + 网络 |

> 💡 **结论**：没有「最好的工具」，只有**最适合你 PDF 类型的工具组合**。


## 0. 安装依赖

In [1]:
%pip install PyPDF2 PyMuPDF pdfplumber -q

Note: you may need to restart the kernel to use updated packages.


In [3]:
import re
from pathlib import Path

import fitz  # PyMuPDF
import pdfplumber

# 修改为你的 PDF 路径
PDF_PATH = Path("doc/attention_is_all_you_need.pdf")
OUTPUT_DIR = Path("out") / "text" / PDF_PATH.stem
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not PDF_PATH.exists():
    raise FileNotFoundError(f"请把 PDF 放到: {PDF_PATH.resolve()}")

## 1. 方法 A — PyPDF2（最朴素）

纯 Python、依赖最少。`page.extract_text()` 一行搞定，但：
- 复杂版面容易乱序
- 表格、多栏支持弱

适合：快速验证「能不能提文本」。

In [4]:
def extract_pypdf2(pdf_path: Path) -> str:
    from PyPDF2 import PdfReader
    reader = PdfReader(pdf_path)
    return "\n".join(p.extract_text() or "" for p in reader.pages)

In [11]:
text_a = extract_pypdf2(PDF_PATH)
print(f"字符数: {len(text_a)}")
print("--- 前 500 字 ---")
print(text_a[20000:25000])

字符数: 39486
--- 前 500 字 ---
 of about 37000 tokens. For English-French, we used the significantly larger WMT
2014 English-French dataset consisting of 36M sentences and split tokens into a 32000 word-piece
vocabulary [ 38]. Sentence pairs were batched together by approximate sequence length. Each training
batch contained a set of sentence pairs containing approximately 25000 source tokens and 25000
target tokens.
5.2 Hardware and Schedule
We trained our models on one machine with 8 NVIDIA P100 GPUs. For our base models using
the hyperparameters described throughout the paper, each training step took about 0.4 seconds. We
trained the base models for a total of 100,000 steps or 12 hours. For our big models,(described on the
bottom line of table 3), step time was 1.0 seconds. The big models were trained for 300,000 steps
(3.5 days).
5.3 Optimizer
We used the Adam optimizer [ 20] with β1= 0.9,β2= 0.98andϵ= 10−9. We varied the learning
rate over the course of training, according to the formu

## 2. 方法 B — PyMuPDF（推荐默认）

速度极快，文本质量通常优于 PyPDF2，API 还能提图片、坐标。

单栏文档直接 `page.get_text()` 即可。

In [6]:
def extract_pymupdf(pdf_path: Path) -> str:
    doc = fitz.open(pdf_path)
    pages = [page.get_text() for page in doc]
    doc.close()
    return "\n\n".join(pages)

In [10]:
text_b = extract_pymupdf(PDF_PATH)
print(f"字符数: {len(text_b)}")
print("--- 前 500 字 ---")
print(text_b[20000:25000])

字符数: 39526
--- 前 500 字 ---
ntences were encoded using byte-pair encoding [3], which has a shared source-
target vocabulary of about 37000 tokens. For English-French, we used the significantly larger WMT
2014 English-French dataset consisting of 36M sentences and split tokens into a 32000 word-piece
vocabulary [38]. Sentence pairs were batched together by approximate sequence length. Each training
batch contained a set of sentence pairs containing approximately 25000 source tokens and 25000
target tokens.
5.2
Hardware and Schedule
We trained our models on one machine with 8 NVIDIA P100 GPUs. For our base models using
the hyperparameters described throughout the paper, each training step took about 0.4 seconds. We
trained the base models for a total of 100,000 steps or 12 hours. For our big models,(described on the
bottom line of table 3), step time was 1.0 seconds. The big models were trained for 300,000 steps
(3.5 days).
5.3
Optimizer
We used the Adam optimizer [20] with β1 = 0.9, β2 =

## 多栏布局：最常见的坑

暴力提取按绘图顺序逐行读取，两栏文本交错——句子被腰斩，语义全乱。

**❌ 暴力提取结果：**
```
左栏第1行  右栏第1行
左栏第2行  右栏第2行
左栏第3行  右栏第3行
```
句子被腰斩、语义全乱

**✅ 正确的顺序：**
```
左栏第1行
左栏第2行
左栏第3行
右栏第1行
右栏第2行
右栏第3行
```

> ⚠️ **陷阱 #2**：两栏文本直接拼 → 句子腰斩


## 多栏算法：按 x 坐标聚类

```python
def reconstruct_columns(words, page_width):
    midpoint = page_width / 2
    left = [w for w in words if w["x0"] < midpoint]
    right = [w for w in words if w["x0"] >= midpoint]
    if len(right) < 0.1 * len(words):
        return sort_by_reading(words)
    return sort_by_reading(left) + "\n\n" + sort_by_reading(right)

def sort_by_reading(words):
    return " ".join(
        w["text"]
        for w in sorted(words, key=lambda w: (round(w["top"], 1), w["x0"]))
    )
```

### 进阶：N 栏自动识别
- 用 K-Means 对所有词的 `x0` 做聚类
- 用 Silhouette score 自动选 k（1栏/2栏/3栏）


## 3. 多栏布局的坑

**问题**：两栏论文直接 `extract_text()` 会得到交错文本：

```
左栏第1行  右栏第1行
左栏第2行  右栏第2行
```

**解决思路**（方法 C）：

1. 用 `page.extract_words()` 拿到每个词的 `(x0, top, text)`
2. 按 `x0` 聚类，识别左栏 / 右栏
3. 每栏内按 `(top, x0)` 排序
4. 先读完左栏，再读右栏

### 3.1 先看一页的 word 坐标

In [ ]:
with pdfplumber.open(PDF_PATH) as pdf:
    page = pdf.pages[0]
    words = page.extract_words()
    page_width = page.width

print(f"第 1 页宽度: {page_width:.1f} pt，共 {len(words)} 个词")
print(f"页宽中点: {page_width / 2:.1f} pt")
print()
print("前 10 个词（text, x0, top）：")
for w in words[:10]:
    side = "左栏" if w["x0"] < page_width / 2 else "右栏"
    print(f"  {w['text']!r:15s} x0={w['x0']:6.1f} top={w['top']:6.1f}  → {side}")

### 3.2 多栏重建函数

In [12]:
def _reconstruct_columns(words: list[dict], page_width: float) -> str:
    """简化版多栏重建：按页宽中点把词分成左右两组。"""
    midpoint = page_width / 2

    left_col = [w for w in words if w["x0"] < midpoint]
    right_col = [w for w in words if w["x0"] >= midpoint]

    # 如果右栏几乎没词，说明是单栏，按原顺序拼即可
    if len(right_col) < 0.1 * len(words):
        return " ".join(w["text"] for w in sorted(words, key=lambda w: (w["top"], w["x0"])))

    def col_to_text(col):
        col_sorted = sorted(col, key=lambda w: (round(w["top"], 1), w["x0"]))
        return " ".join(w["text"] for w in col_sorted)

    return col_to_text(left_col) + "\n\n" + col_to_text(right_col)

In [13]:
def extract_multicolumn(pdf_path: Path) -> str:
    """处理两栏 / 多栏论文。"""
    all_text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, 1):
            words = page.extract_words()
            if not words:
                continue
            text = _reconstruct_columns(words, page.width)
            all_text.append(f"--- Page {page_num} ---\n{text}")
    return "\n\n".join(all_text)

In [14]:
text_c = extract_multicolumn(PDF_PATH)
print(f"字符数: {len(text_c)}")
print("--- 前 500 字 ---")
print(text_c[:500])

字符数: 35782
--- 前 500 字 ---
--- Page 1 ---
Providedproperattributionisprovided,Googleherebygrantspermissionto reproducethetablesandfiguresinthispapersolelyforuseinjournalisticor scholarlyworks. Attention Is All 3202 AshishVaswani∗ NoamShazeer∗ GoogleBrain GoogleBrain avaswani@google.com noam@google.com guA LlionJones∗ AidanN.Gomez∗ GoogleResearch UniversityofToronto 2 llion@google.com aidan@cs.toronto.edu ]LC.sc[ IlliaPolosukhin∗ illia.polosukhin@gmail.com Abstract 7v26730.6071:viXra Thedominantsequencetransductionmodelsar


## 4. 三种方法对比

In [15]:
print(f"📊 字符数对比：")
print(f"  方法 A (PyPDF2):     {len(text_a):>6}")
print(f"  方法 B (PyMuPDF):    {len(text_b):>6}")
print(f"  方法 C (多栏):       {len(text_c):>6}")

(OUTPUT_DIR / "pypdf2.txt").write_text(text_a, encoding="utf-8")
(OUTPUT_DIR / "pymupdf.txt").write_text(text_b, encoding="utf-8")
(OUTPUT_DIR / "multicolumn.txt").write_text(text_c, encoding="utf-8")
print(f"\n✅ 已保存到 {OUTPUT_DIR}")

📊 字符数对比：
  方法 A (PyPDF2):      39486
  方法 B (PyMuPDF):     39526
  方法 C (多栏):        35782


## 文本清洗：让 LLM 能正确理解

### 4 个必做清洗

```python
import re

def clean(text):
    # 1. 合并行尾连字符断词: informa-\ntion → information
    text = re.sub(r"-\n(\w)", r"\1", text)
    # 2. 段内换行合并（单个 \n → 空格）
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    # 3. 压多重空白
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    # 4. 去全角空格、零宽字符
    text = text.replace("　", " ").replace("\u200b", "")
    return text.strip()
```

> ⚠️ **陷阱 #5**：跨行连字符没合并 → 检索不到完整单词


## 5. 文本清洗

原始提取结果常有这些噪音，入库前必须处理：

| 问题 | 示例 | 处理 |
|------|------|------|
| 行尾连字符断词 | `informa-\ntion` | 合并为 `information` |
| 段内换行 | 同一段被硬换行 | 换行 → 空格 |
| 多余空白 | 多个空格/Tab | 压缩为一个空格 |
| 过多空行 | 3+ 连续换行 | 压缩为两个换行 |

In [16]:
def clean_text(text: str) -> str:
    # 1. 合并行尾连字符断词：informa-\ntion -> information
    text = re.sub(r"-\n(\w)", r"\1", text)

    # 2. 把单个换行（同一段内）替换为空格
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)

    # 3. 多个空白压缩为一个空格
    text = re.sub(r"[ \t]+", " ", text)

    # 4. 多个空行压缩为两个换行
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [17]:
cleaned = clean_text(text_b)
print(f"清洗前: {len(text_b)} 字符 → 清洗后: {len(cleaned)} 字符")
print("--- 清洗后前 500 字 ---")
print(cleaned[:500])

(OUTPUT_DIR / "pymupdf_cleaned.txt").write_text(cleaned, encoding="utf-8")
print(f"✅ 清洗结果 → {OUTPUT_DIR / 'pymupdf_cleaned.txt'}")

清洗前: 39526 字符 → 清洗后: 39461 字符
--- 清洗后前 500 字 ---
Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗† University of Toronto aidan@cs.toronto.edu Łukasz K


## 去页眉页脚算法

**思路**：跨多页**重复出现**的首行/尾行 → 大概率是页眉页脚。

```python
from collections import Counter

def remove_headers_footers(pages):
    first_lines = [p.split("\n")[0].strip() for p in pages if p]
    last_lines = [p.rsplit("\n")[-1].strip() for p in pages if p]
    threshold = len(pages) // 2
    headers = {line for line, n in Counter(first_lines).items()
               if n >= threshold and len(line) < 80}
    footers = {line for line, n in Counter(last_lines).items()
               if n >= threshold and len(line) < 80}
    cleaned = []
    for p in pages:
        lines = p.split("\n")
        if lines and lines[0].strip() in headers:
            lines = lines[1:]
        if lines and lines[-1].strip() in footers:
            lines = lines[:-1]
        cleaned.append("\n".join(lines))
    return cleaned
```

> ⚠️ **陷阱 #4**：页眉页脚没去 → 每个 chunk 带噪音


## 6. 页眉页脚去除

**思路**：跨页找**完全相同**的首行 / 尾行，出现在过半数页面上的短行，大概率是页眉页脚。

例如：`第 3 页`、`Company Confidential` 会在每页重复出现。

In [18]:
def remove_headers_footers(pages: list[str]) -> list[str]:
    """跨页找完全相同的首/尾行，认定为页眉页脚去掉。"""
    if len(pages) < 3:
        return pages

    first_lines = [p.split("\n", 1)[0].strip() for p in pages if p]
    last_lines = [p.rsplit("\n", 1)[-1].strip() for p in pages if p]

    threshold = len(pages) // 2

    def common(lines):
        from collections import Counter
        c = Counter(lines)
        return {line for line, count in c.items() if count >= threshold and len(line) < 80}

    headers = common(first_lines)
    footers = common(last_lines)

    cleaned = []
    for p in pages:
        lines = p.split("\n")
        if lines and lines[0].strip() in headers:
            lines = lines[1:]
        if lines and lines[-1].strip() in footers:
            lines = lines[:-1]
        cleaned.append("\n".join(lines))
    return cleaned

In [19]:
# 演示：按页拆分后去页眉页脚
pages_raw = text_b.split("\n\n")
pages_clean = remove_headers_footers(pages_raw)

print(f"共 {len(pages_raw)} 页")
print(f"去页眉页脚后总字符: {sum(len(p) for p in pages_clean)}")

共 15 页
去页眉页脚后总字符: 39484


## 7. 小结 & 下一步

| 场景 | 推荐方法 |
|------|----------|
| 普通单栏文档 | 方法 B（PyMuPDF）+ `clean_text` |
| 两栏论文 | 方法 C（pdfplumber 多栏） |
| 含表格 | → [03_extract_tables.py](03_extract_tables.py) |
| 扫描页 | → [04_ocr_scanned.py](04_ocr_scanned.py) |

**课堂练习**：找一份两栏 PDF，对比方法 B 和方法 C 的输出差异。